## Imports & basic config

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction import DictVectorizer
import pickle

PROJECT_ROOT = Path("..").resolve().parents[0]  # adjust if needed
DATA_DIR = PROJECT_ROOT / "01-intro" / "data"

DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version)
print("Pandas:", pd.__version__)


Python: 3.13.1 (main, Dec  3 2024, 17:59:52) [Clang 16.0.0 (clang-1600.0.26.4)]
Pandas: 2.3.3


## Helper: download data (if you want it self-contained)

Later, you’ll follow exactly the URLs from the homework (in cohorts/<year>/01-intro/homework.md), but this is the structure:

[💽 NYC Taxi Data source](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.pag)

In [3]:
import urllib.request

def download_file(url: str, dest_path: Path):
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    if dest_path.exists():
        print(f"✅ Already exists: {dest_path.name}")
        return
    print(f"⬇️ Downloading {url} -> {dest_path}")
    urllib.request.urlretrieve(url, dest_path)
    print("Done.")

# Example placeholders – replace with the actual links from the homework
green_tripdata_2024_01 = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2024-01.parquet"
green_tripdata_2024_02 = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2024-02.parquet"

download_file(green_tripdata_2024_01, DATA_DIR / "green_tripdata_2024-01.parquet")
download_file(green_tripdata_2024_02, DATA_DIR / "green_tripdata_2024-02.parquet")


⬇️ Downloading https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2024-01.parquet -> /Users/jonathan/Documents/self_trainings/mlops/mlops-zoomcamp-training/01-intro/data/green_tripdata_2024-01.parquet
Done.
⬇️ Downloading https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2024-02.parquet -> /Users/jonathan/Documents/self_trainings/mlops/mlops-zoomcamp-training/01-intro/data/green_tripdata_2024-02.parquet
Done.


## Load data & basic cleaning

In [4]:
def read_data(path: Path):
    df = pd.read_parquet(path)
    
    # Example feature used in the course: duration in minutes
    df["duration"] = (df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]).dt.total_seconds() / 60

    # Filter out extreme values
    df = df[(df["duration"] >= 1) & (df["duration"] <= 60)].copy()

    # Convert categorical features to string
    categorical = ["PULocationID", "DOLocationID"]
    for col in categorical:
        df[col] = df[col].astype(str)

    return df


train_path = DATA_DIR / "green_tripdata_2024-01.parquet"
val_path = DATA_DIR / "green_tripdata_2024-02.parquet"

df_train = read_data(train_path)
df_val = read_data(val_path)

len(df_train), len(df_val)


(54373, 51497)

In [5]:
df_train.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,duration
0,2,2024-01-01 00:46:55,2024-01-01 00:58:25,N,1.0,236,239,1.0,1.98,12.8,...,0.5,3.61,0.0,NaN,1.0,21.66,1.0,1.0,2.75,11.500000
1,2,2024-01-01 00:31:42,2024-01-01 00:52:34,N,1.0,65,170,5.0,6.54,30.3,...,0.5,7.11,0.0,NaN,1.0,42.66,1.0,1.0,2.75,20.866667
2,2,2024-01-01 00:30:21,2024-01-01 00:49:23,N,1.0,74,262,1.0,3.08,19.8,...,0.5,3.00,0.0,NaN,1.0,28.05,1.0,1.0,2.75,19.033333
3,1,2024-01-01 00:30:20,2024-01-01 00:42:12,N,1.0,74,116,1.0,2.40,14.2,...,1.5,0.00,0.0,NaN,1.0,16.70,2.0,1.0,0.00,11.866667
4,2,2024-01-01 00:32:38,2024-01-01 00:43:37,N,1.0,74,243,1.0,5.14,22.6,...,0.5,6.28,0.0,NaN,1.0,31.38,1.0,1.0,0.00,10.983333


## Prepare features / target

In [6]:
features = ["PULocationID", "DOLocationID", "trip_distance"]
target = "duration"

def get_feature_target(df: pd.DataFrame):
    dicts = df[features].to_dict(orient="records")
    y = df[target].values
    return dicts, y

train_dicts, y_train = get_feature_target(df_train)
val_dicts, y_val = get_feature_target(df_val)

dv = DictVectorizer()

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

X_train.shape, X_val.shape


((54373, 449), (51497, 449))

## Train baseline model & evaluate

In [8]:
help(mean_squared_error)

Help on function mean_squared_error in module sklearn.metrics._regression:

mean_squared_error(
    y_true,
    y_pred,
    *,
    sample_weight=None,
    multioutput='uniform_average'
)
    Mean squared error regression loss.

    Read more in the :ref:`User Guide <mean_squared_error>`.

    Parameters
    ----------
    y_true : array-like of shape (n_samples,) or (n_samples, n_outputs)
        Ground truth (correct) target values.

    y_pred : array-like of shape (n_samples,) or (n_samples, n_outputs)
        Estimated target values.

    sample_weight : array-like of shape (n_samples,), default=None
        Sample weights.

    multioutput : {'raw_values', 'uniform_average'} or array-like of shape             (n_outputs,), default='uniform_average'
        Defines aggregating of multiple output values.
        Array-like value defines weights used to average errors.

        'raw_values' :
            Returns a full set of errors in case of multioutput input.

        'uniform_ave

In [9]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

rmse = mean_squared_error(y_val, y_pred)
print(f"Validation RMSE: {rmse:.2f} minutes")


Validation RMSE: 52.77 minutes


## Save model + DictVectorizer (first step toward MLOps)

In [10]:
MODEL_DIR = PROJECT_ROOT / "01-intro" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "linreg_model.bin"

with open(model_path, "wb") as f_out:
    pickle.dump((dv, lr), f_out)

model_path


PosixPath('/Users/jonathan/Documents/self_trainings/mlops/mlops-zoomcamp-training/01-intro/models/linreg_model.bin')